# Sales
1. Define schema
2. Load CSV
3. Populate Country field with "USA"
4. Load InternationalSales into a dataframe, and append
5. Add the ZipCountry column as text
6. Write to Delta table


## 1. Schema and 2. Load

In [1]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DecimalType

# Define the schema
# True indicates the field is nullable
salesschema = StructType([
    StructField("ProductID", IntegerType(), True),
    StructField("Date", DateType(), True),
    StructField("Zip", StringType(), True),
    StructField("Units", IntegerType(), True),
    StructField("Revenue", DecimalType(10, 2), True),
    StructField("Country", StringType(), True)
])


# Load the CSV file with the defined schema
df = spark.read.csv("Files/USSales/Sales.csv", schema=salesschema, header=True)

# Show the data
df.show()

StatementMeta(, f73779b4-fbe2-4bd0-b6db-d35bf733f41b, 3, Finished, Available, Finished)

+---------+----------+-----+-----+-------+-------+
|ProductID|      Date|  Zip|Units|Revenue|Country|
+---------+----------+-----+-----+-------+-------+
|     1076|2015-01-20|72638|    1| 254.57|   NULL|
|     1076|2015-01-21|47577|    1| 254.57|   NULL|
|     1076|2015-01-28|34653|    1| 254.57|   NULL|
|     1076|2015-01-31|84014|    1| 254.57|   NULL|
|     1076|2015-02-01|75070|    1| 254.57|   NULL|
|     1076|2015-02-01|87031|    1| 254.57|   NULL|
|     1076|2015-02-03|72019|    1| 254.57|   NULL|
|     1076|2015-02-03|72086|    1| 254.57|   NULL|
|     1076|2015-02-03|77089|    2| 509.15|   NULL|
|     1076|2015-02-09|07649|    1| 254.57|   NULL|
|     1076|2015-02-11|79705|    1| 254.57|   NULL|
|     1076|2015-02-14|92624|    1| 254.57|   NULL|
|     1076|2015-02-22|08527|    1| 254.57|   NULL|
|     1076|2015-02-22|08816|    1| 254.57|   NULL|
|     1076|2015-02-23|24740|    1| 254.57|   NULL|
|     1076|2015-02-24|63023|    1| 254.57|   NULL|
|     1076|2015-02-25|32503|   

## Verify the load worked properly
* Quickly double-check the file

In [2]:
# Did we get all the data? 

row_count = df.count()
print(f"Number of rows: {row_count}")

distinct_countries = df.select("Zip").distinct().show()

StatementMeta(, f73779b4-fbe2-4bd0-b6db-d35bf733f41b, 4, Finished, Available, Finished)

Number of rows: 4198753
+-----+
|  Zip|
+-----+
|91910|
|35004|
|39350|
|37311|
|75602|
|97128|
|85022|
|32773|
|37774|
|42743|
|32812|
|06382|
|08648|
|65548|
|92027|
|95519|
|46534|
|94102|
|77339|
|35640|
+-----+
only showing top 20 rows



## 3. Populate the "USA" values in Country column

In [3]:
# Populate the Country column with USA

from pyspark.sql.functions import lit

# Populate the 'Country' field with "USA"
df = df.withColumn("Country", lit("USA"))

# Show the updated data
df.show()

StatementMeta(, f73779b4-fbe2-4bd0-b6db-d35bf733f41b, 5, Finished, Available, Finished)

+---------+----------+-----+-----+-------+-------+
|ProductID|      Date|  Zip|Units|Revenue|Country|
+---------+----------+-----+-----+-------+-------+
|     1076|2015-01-20|72638|    1| 254.57|    USA|
|     1076|2015-01-21|47577|    1| 254.57|    USA|
|     1076|2015-01-28|34653|    1| 254.57|    USA|
|     1076|2015-01-31|84014|    1| 254.57|    USA|
|     1076|2015-02-01|75070|    1| 254.57|    USA|
|     1076|2015-02-01|87031|    1| 254.57|    USA|
|     1076|2015-02-03|72019|    1| 254.57|    USA|
|     1076|2015-02-03|72086|    1| 254.57|    USA|
|     1076|2015-02-03|77089|    2| 509.15|    USA|
|     1076|2015-02-09|07649|    1| 254.57|    USA|
|     1076|2015-02-11|79705|    1| 254.57|    USA|
|     1076|2015-02-14|92624|    1| 254.57|    USA|
|     1076|2015-02-22|08527|    1| 254.57|    USA|
|     1076|2015-02-22|08816|    1| 254.57|    USA|
|     1076|2015-02-23|24740|    1| 254.57|    USA|
|     1076|2015-02-24|63023|    1| 254.57|    USA|
|     1076|2015-02-25|32503|   

## 4. Append the InternationalSales table to the Sales table
1. Load the df_intl dataframe

In [4]:
# Load the InternationalSales table into a second dataframe

df_intl = spark.sql("SELECT * FROM DIAD_PY.internationalsales")
display(df_intl)

StatementMeta(, f73779b4-fbe2-4bd0-b6db-d35bf733f41b, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 11a9a6ed-aa6a-4fc9-a17d-142a6a0b829f)

In [5]:
# Append the df_intl dataframe to the df dataframe
# Union only works here because both df and df_intl have the exact same schema

df = df.union(df_intl)

display(df)

StatementMeta(, f73779b4-fbe2-4bd0-b6db-d35bf733f41b, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 75e59c26-e4e4-4a99-b687-79adb5961321)

## Verify the Union worked

In [6]:
# Did we get all the countries? 

row_count = df.count()
print(f"Number of rows: {row_count}")

distinct_countries = df.select("Country").distinct().show()

StatementMeta(, f73779b4-fbe2-4bd0-b6db-d35bf733f41b, 8, Finished, Available, Finished)

Number of rows: 7235491
+---------+
|  Country|
+---------+
|      USA|
|  Germany|
|  Nigeria|
|   Mexico|
|   Canada|
|    Japan|
|  Country|
|Australia|
+---------+



## 5. Add the Calculated column ZipCountry

In [7]:
# Adding the ZipCountry text column

from pyspark.sql.functions import concat_ws

df = df.withColumn("ZipCountry", concat_ws(",", df["Zip"], df["Country"]))


StatementMeta(, f73779b4-fbe2-4bd0-b6db-d35bf733f41b, 9, Finished, Available, Finished)

In [8]:
# Verifying the results of the new column

df.select("Zip", "Country", "ZipCountry").show(5)


StatementMeta(, f73779b4-fbe2-4bd0-b6db-d35bf733f41b, 10, Finished, Available, Finished)

+-----+-------+----------+
|  Zip|Country|ZipCountry|
+-----+-------+----------+
|72638|    USA| 72638,USA|
|47577|    USA| 47577,USA|
|34653|    USA| 34653,USA|
|84014|    USA| 84014,USA|
|75070|    USA| 75070,USA|
+-----+-------+----------+
only showing top 5 rows



## 6. Write the Delta table

In [9]:
# Let's write the Delta table

df.write.format("delta").mode("overwrite").saveAsTable("Sales")

StatementMeta(, f73779b4-fbe2-4bd0-b6db-d35bf733f41b, 11, Finished, Available, Finished)